# Slider 2v2 신경망(NNUE) 학습 (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Perdume/Slider/blob/rust-port/colab/slider_nnue_training.ipynb)

기존 선형 평가 위에 **작은 신경망(147 입력 → 은닉 32 → 출력 1)** 을 더해 학습합니다. 평가 = 선형 평가 + 신경망 출력이라서, 학습 전에는 기존 평가와 같습니다.
엔진과 학습기는 모두 `rust/` 의 Rust 코드이고, 학습도 외부 라이브러리 없이 직접 구현한 역전파(Adam)입니다. GPU 는 필요 없습니다.

1. Rust 를 설치하고 레포를 받아 빌드합니다.
2. 현재 평가(선형 + 지금까지 채택한 신경망)로 AI 끼리 자가대전을 두고, 국면(타일/내 말/상대 말), 평가 특징, 탐색 점수, 최종 결과를 JSONL 로 기록합니다.
3. 모은 데이터 전체로 신경망을 처음부터 학습합니다 (`trainnet`). 목표는 `λ·sigmoid(탐색점수/K) + (1-λ)·판 결과`, 판 단위 80/20 검증에서 손실이 가장 낮았던 시점을 저장합니다.
4. **새 신경망 vs 현재 평가** 대국으로 정말 강해졌는지 확인하고, 이기면 채택합니다.
5. 1~4 를 `ROUNDS` 번 반복하고 `net.txt` 를 내려받습니다.

> 학습 로그의 "검증 손실: 선형 평가만"과 "가장 좋은 검증 손실"을 비교해 보세요. 신경망이 예측을 못 개선하면(0% 근처) 데이터를 더 모으거나(`GAMES_PER_ROUND`) 수당 시간을 늘려야 합니다.
> 이 게임은 무승부가 많아 승패 신호가 약합니다. 손실이 줄어도 대국에서 강해지지 않을 수 있어서 4번 확인 단계가 있습니다.

## 1. 설정

In [ ]:
import os, sys, json, re, shutil, subprocess, pathlib

# ===== 레포 =====
REPO_URL = "https://github.com/Perdume/Slider.git"
BRANCH = "rust-port"   # Rust 코드가 master 에 합쳐지면 "master" 로 바꾸세요

# ===== 학습 설정 (처음엔 그대로 두고, 시간이 남으면 늘려 보세요) =====
ROUNDS = 2             # "자가대전 -> 학습 -> 대국 확인" 반복 횟수
GAMES_PER_ROUND = 600  # 라운드마다 자가대전 판 수 (한 판에 국면 약 100개)
TIME_MS = 30           # 자가대전에서 수당 탐색 시간(ms)
RANDOM_PLIES = 4       # 판 시작의 무작위 수 (시작 국면 다양화)
EPSILON = 0.05         # 그 뒤에도 이 확률로 무작위 수를 둠 (데이터 다양화)
EPOCHS = 40            # 신경망 학습 epoch 수
MATCH_GAMES = 200      # 라운드 끝에 새 신경망 vs 현재 평가 대국 수 (짝수)
MATCH_TIME_MS = 50
SEED = 1
LAMBDA = 1.0           # 학습 목표에서 탐색 점수의 비중 (0 = 판 결과만, 1 = 탐색 점수만). 로컬 시험에서 1.0 이 0.7 보다 대국 성적이 좋았습니다
ACCEPT_SCORE = 0.55    # 대국 점수가 이 값 이상이면 채택 (0.5 = 동률)

# 튜닝된 선형 가중치(weights.json)를 쓰려면 Colab 왼쪽 파일 탭에 올리고 경로를 적으세요. None 이면 기본 가중치.
WEIGHTS_PATH = None

THREADS = os.cpu_count() or 2
WORK_DIR = pathlib.Path(os.environ.get("SLIDER_WORK_DIR", "/content/slider_nn_work"))
DATA_DIR = WORK_DIR / "data"
for d in (WORK_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)
BASE_WEIGHTS = WEIGHTS_PATH if WEIGHTS_PATH and os.path.exists(WEIGHTS_PATH) else "default"
if WEIGHTS_PATH and BASE_WEIGHTS == "default":
    print("경고: WEIGHTS_PATH 파일이 없어서 기본 가중치를 씁니다:", WEIGHTS_PATH)
print("work dir:", WORK_DIR, "| threads:", THREADS, "| 선형 가중치:", BASE_WEIGHTS)

## 2. Rust 설치 · 레포 받기 · 빌드

In [ ]:
def sh(cmd, **kw):
    print("$", " ".join(map(str, cmd)) if isinstance(cmd, list) else cmd)
    subprocess.run(cmd, shell=isinstance(cmd, str), check=True, **kw)

os.environ["PATH"] = str(pathlib.Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]
if shutil.which("cargo") is None:
    sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")

# SLIDER_REPO_DIR 환경변수로 이미 받아 둔 레포를 쓸 수도 있습니다 (로컬 실행용)
if os.environ.get("SLIDER_REPO_DIR"):
    REPO_DIR = pathlib.Path(os.environ["SLIDER_REPO_DIR"])
else:
    REPO_DIR = WORK_DIR / "Slider"
    if REPO_DIR.exists():
        sh(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    else:
        sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])

RUST_DIR = REPO_DIR / "rust"
if not (RUST_DIR / "Cargo.toml").exists():
    raise RuntimeError(
        f"받아온 레포에 rust/Cargo.toml 이 없습니다 (BRANCH={BRANCH!r}). Rust 포팅이 올라간 브랜치를 쓰고, "
        "이미 받아 둔 폴더가 있으면 지우고 다시 실행하세요: " + str(REPO_DIR))
sh(["cargo", "build", "--release"], cwd=RUST_DIR)
BIN = RUST_DIR / "target" / "release"
print("빌드 완료:", sorted(p.name for p in BIN.iterdir() if p.is_file() and os.access(p, os.X_OK) and "." not in p.name))

## 3. 실행 함수

In [ ]:
def run(binary, args):
    """Rust 실행 파일을 돌리고 출력을 실시간으로 보여준 뒤 줄 목록을 돌려줍니다."""
    cmd = [str(BIN / binary), *map(str, args)]
    print("$", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        line = line.rstrip("\n")
        lines.append(line)
        print("   ", line)
    if proc.wait() != 0:
        raise RuntimeError(f"{binary} 실행 실패 (exit {proc.returncode})")
    return lines


def run_selfplay(out_path, games, seed, net_path):
    args = ["--games", games, "--time-ms", TIME_MS, "--random-plies", RANDOM_PLIES, "--epsilon", EPSILON,
            "--threads", THREADS, "--seed", seed, "--weights", BASE_WEIGHTS, "--positions", "yes", "--out", out_path]
    if net_path:
        args += ["--net", net_path]
    return run("selfplay", args)


def run_train(data_files, out_path):
    return run("trainnet", ["--data", ",".join(map(str, data_files)), "--weights", BASE_WEIGHTS, "--out", out_path,
                            "--epochs", EPOCHS, "--lambda", LAMBDA, "--seed", SEED])


def run_match(new_net, current_net):
    args = ["--a", BASE_WEIGHTS, "--net-a", new_net, "--b", BASE_WEIGHTS, "--games", MATCH_GAMES,
            "--time-ms", MATCH_TIME_MS, "--threads", THREADS, "--seed", SEED + 1000]
    if current_net:
        args += ["--net-b", current_net]
    lines = run("matchup", args)
    result = [l for l in lines if l.startswith("RESULT ")]
    return json.loads(result[-1][len("RESULT "):])

## 4. 학습 루프
라운드마다: 자가대전 → (지금까지 모은 데이터 전체로) 신경망 학습 → 새 신경망 vs 현재 평가 대국.
대국 점수가 `ACCEPT_SCORE` 이상일 때만 새 신경망을 채택합니다. 채택하지 못하면 다음 라운드는 같은 평가로 데이터를 더 모읍니다.

In [ ]:
current_net = None      # 지금까지 채택한 신경망 파일 (None = 선형 평가만)
data_files = []
history = []

for rnd in range(1, ROUNDS + 1):
    print(f"\n=============== 라운드 {rnd}/{ROUNDS} ===============")
    print(f"[1/3] 자가대전 {GAMES_PER_ROUND}판 (현재 평가: 선형 + {'신경망' if current_net else '신경망 없음'})")
    out = DATA_DIR / f"selfplay_round{rnd}.jsonl"
    run_selfplay(out, GAMES_PER_ROUND, SEED + rnd, current_net)
    data_files.append(out)

    print(f"[2/3] 신경망 학습 (데이터 {len(data_files)}개 파일 전체)")
    candidate = WORK_DIR / f"net_round{rnd}.txt"
    run_train(data_files, candidate)

    print(f"[3/3] 대국 확인: 새 신경망 vs {'지금까지 채택한 평가' if current_net else '선형 평가만'}")
    r = run_match(candidate, current_net)
    accepted = r["score"] >= ACCEPT_SCORE
    history.append({"round": rnd, **r, "accepted": accepted})
    print(f"    -> 새 신경망 {r['aWins']}승 {r['draws']}무 {r['bWins']}패, 점수 {r['score']:.3f} "
          f"(95% 구간 {r['low']:.3f} ~ {r['high']:.3f}) => {'채택' if accepted else '채택 안 함'}")
    if accepted:
        current_net = candidate

print("\n라운드 요약")
for h in history:
    print(f"  라운드 {h['round']}: {h['aWins']}승 {h['draws']}무 {h['bWins']}패  점수 {h['score']:.3f}  {'채택' if h['accepted'] else '-'}")

## 5. 결과 저장

In [ ]:
final = WORK_DIR / "net.txt"
if current_net:
    shutil.copy(current_net, final)
    print("저장:", final)
    print("게임/대국에 적용: 이 파일을 받아서")
    print("  rust/target/release/matchup --a default --net-a net.txt --b default --games 100")
    try:
        from google.colab import files
        files.download(str(final))
    except ImportError:
        pass
else:
    print("채택된 신경망이 없습니다. 판 수(GAMES_PER_ROUND)나 수당 시간(TIME_MS)을 늘려서 다시 돌려 보세요.")